# Pulse Width Modulation

Simulation results for the pulse width modulation appendix.

For an integrator and a first-order low-pass filter, the average output of the pulse
width modulator is simulated as a function of the static input and characterized by
the Taylor coefficients of the average output and by the resulting third-order output
intercept point. The simulated values are compared with the exact expressions and with
the general filter approximation.

In [ ]:
import os

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from delta_sigma_simulator.filter import (
    FilterButterworth,
    FilterIntegrator,
    FilterFirstOrder,
)
from delta_sigma_simulator.quantizer import QuantizerDelayHysteresis
from delta_sigma_simulator.wave import BinaryWave, SineWave

In [ ]:
# - Directory the generated CSV files are written to. Set the BOOK_TIKZ environment
#   variable to the assets/tikz directory of the book to update its figures directly.
out = Path(os.environ.get("BOOK_TIKZ", "."))
# - Switching frequency
f0 = 1.0
# - Ratio of the corner frequency of the filter to the switching frequency, for the
#   simulated and for the predicted values
beta_sim = np.logspace(-3, 0, 20)
beta = np.logspace(-3, 0, 100)
# - Amplitude of the average output used to extract the Taylor coefficients, and the
#   number of points it is sampled with
W_max = 0.1
n_W = 21

## Average output

The output of the pulse width modulator toggles whenever the filtered carrier crosses
the static input. The zero crossings are located with the quantizer of the simulator,
after which the average output follows from the duty cycle. The filter is driven by a
square wave of frequency `f0` for a number of periods that is long enough for its
transient to have decayed.

In [ ]:
def carrier(h, beta):
    """
    Response of the filter to the carrier, a square wave of frequency f0. The carrier
    covers enough periods for the transient of the filter to have decayed by the end.

    Arguments:
        h: The filter to apply to the carrier.
        beta: Ratio of the corner frequency of the filter to the switching frequency.
    """
    n = int(np.ceil(20 / (2 * np.pi * beta))) + 2

    return BinaryWave(np.arange(0, n + 0.5, 0.5) / f0).filter(h), n - 1


def average_output(y, t0, U):
    """
    Average output of the pulse width modulator for a static input, obtained from the
    two crossings of the filtered carrier with the static input that follow t0.

    Arguments:
        y: Response of the filter to the carrier.
        t0: Start of the period the crossings are located in, at which the filtered
            carrier is at its minimum.
        U: The static input.
    """
    # - A filter without DC feedback offsets the response to the carrier, so the offset
    #   is removed before comparing with the static input
    t = t0 + np.linspace(0, 1 / f0, 1001)
    y0 = (np.max(y(t)) + np.min(y(t))) / 2

    quantizer = QuantizerDelayHysteresis(0.0, 0.0)
    quantizer.t_step = 0.1 / f0

    t = t0
    e = []

    # - The output falls at the first crossing and rises at the second one
    for _ in range(2):
        t += quantizer.next(lambda dt, t=t: U - (y(t + dt) - y0))

        e.append(t)

    return 1 - 2 * f0 * (e[1] - e[0])


def dc_characteristic(y, t0, U):
    """
    Simulate the average output of the pulse width modulator for a range of static
    inputs.

    Arguments:
        y: Response of the filter to the carrier.
        t0: Start of the period the crossings are located in.
        U: Array of static inputs.
    """
    W = np.zeros_like(U)

    for i, U_i in enumerate(U):
        W[i] = average_output(y, t0, U_i)

    return W


def taylor(x, y, n=7):
    """
    Return the first n + 1 Taylor coefficients of y as a function of x, obtained by
    fitting a polynomial of degree n. The abscissa is normalized before fitting to keep
    the system well conditioned.

    Arguments:
        x: Array of abscissa values, centred around zero.
        y: Array of ordinate values.
        n: Degree of the polynomial.
    """
    x0 = np.max(np.abs(x))

    c = np.polynomial.polynomial.polyfit(x / x0, y, n)

    return c / x0 ** np.arange(n + 1)


def oip3(W_1, W_3):
    """
    Third-order output intercept point in decibels.
    """
    return 20 * np.log10(np.sqrt(2 * np.abs(W_1**3 / W_3)))

## Integrator

The filter is an integrator with a gain `2 / pi` at the switching frequency, so that
the modulator has unity gain. The average output is predicted to be a perfectly linear
function of the static input.

In [ ]:
# - Gain of the filter at the switching frequency
A_0 = 2 / np.pi

y, n = carrier(FilterIntegrator(2 * np.pi * f0 * A_0), 1.0)

U = W_max * np.linspace(-1, 1, n_W)

W = dc_characteristic(y, n / f0, U)

c_int_sim = taylor(U, W)

print(f"integrator: W_1 = {c_int_sim[1]:.15f}, W_3 = {c_int_sim[3]:+.3e}")

In [ ]:
plt.plot(U, W - c_int_sim[1] * U, "k-")
plt.xlabel(r"$U$")
plt.ylabel(r"$W - W_1 U$")
plt.grid(which="both", ls="--", alpha=0.5)
plt.show()

## First-order low-pass filter

The filter is a first-order low-pass filter with a DC gain `2 / (pi beta)`, so that the
modulator has unity gain, and a corner frequency `beta f0`. Besides the exact
prediction, the prediction of the general filter approximation is evaluated as well,
for which the gain at the switching frequency is `beta` times the DC gain.

In [ ]:
# - Exact prediction
W_1 = np.ones_like(beta)
W_3 = np.pi**2 * beta**2 / 12

# - Prediction of the general filter approximation
a = 1 + (np.pi * beta / 2) ** 2

W_1_gen = 1 / np.sqrt(a)
W_3_gen = np.pi**2 * beta**2 / (8 * a ** (5 / 2))

c_lpf = np.stack(
    [
        beta,
        W_1,
        W_3,
        oip3(W_1, W_3),
        W_1_gen,
        W_3_gen,
        oip3(W_1_gen, W_3_gen),
    ]
)

c_lpf_sim = np.zeros((4, len(beta_sim)))

for i, beta_i in enumerate(beta_sim):
    # - DC gain that yields unity gain of the modulator
    A_DC = 2 / (np.pi * beta_i)

    y, n = carrier(FilterFirstOrder(beta_i * f0, A_DC), beta_i)

    U = W_max * np.linspace(-1, 1, n_W)

    W = dc_characteristic(y, n / f0, U)

    c = taylor(U, W)

    c_lpf_sim[:, i] = beta_i, c[1], c[3], oip3(c[1], c[3])

np.savetxt(
    out / "pwm-evaluation.csv",
    c_lpf.T,
    delimiter=",",
    header="beta,W1_lpf,W3_lpf,OIP3_lpf,W1_gen,W3_gen,OIP3_gen",
    comments="",
)
np.savetxt(
    out / "pwm-evaluation-simulation.csv",
    c_lpf_sim.T,
    delimiter=",",
    header="beta,W1_sim,W3_sim,OIP3_sim",
    comments="",
)

In [ ]:
plt.loglog(beta, c_lpf[1], "k-", label=r"$W_1$")
plt.loglog(beta, c_lpf[4], "k--", label=r"$W_{1,\mathrm{gen}}$")
plt.loglog(beta_sim, c_lpf_sim[1], "ko", label=r"$W_{1,\mathrm{sim}}$")
plt.loglog(beta, c_lpf[2], "-", color="gray", label=r"$W_3$")
plt.loglog(beta, c_lpf[5], "--", color="gray", label=r"$W_{3,\mathrm{gen}}$")
plt.loglog(beta_sim, c_lpf_sim[2], "o", color="gray", label=r"$W_{3,\mathrm{sim}}$")
plt.xlabel(r"$\beta$")
plt.grid(which="both", ls="--", alpha=0.5)
plt.legend()
plt.show()

In [ ]:
plt.semilogx(beta, c_lpf[3], "k-", label=r"$A_\mathrm{oip3}$")
plt.semilogx(beta, c_lpf[6], "k--", label=r"$A_\mathrm{oip3,gen}$")
plt.semilogx(beta_sim, c_lpf_sim[3], "ko", label=r"$A_\mathrm{oip3,sim}$")
plt.xlabel(r"$\beta$")
plt.ylabel(r"$A_\mathrm{oip3}$ [dB]")
plt.grid(which="both", ls="--", alpha=0.5)
plt.legend()
plt.show()

## Input frequency

The evaluation above uses a static input. To verify that the nonlinearity of the
modulator does not depend on the input frequency, the static input is replaced by a
sinusoidal input of frequency `f_u` and amplitude `U`, and the third-order intercept
point is obtained from the demodulated output: the output is low-pass filtered by a
tenth-order Butterworth filter with a corner frequency of `4 f_u`, sampled at sixteen
times that corner frequency, and the fundamental and the third harmonic are read from
its discrete Fourier transform.

The output of the modulator is periodic with the input as long as the input period
spans an integer number `K` of carrier periods, so the transitions of a single period
are located once and then repeated to cover both the settling of the demodulation
filter and the evaluated window.

In [ ]:
def modulate(h, f_u, U, n_carrier):
    """
    Transitions of the pulse width modulator over one period of a sinusoidal input, and
    the value of the output at the start of that period.

    Arguments:
        h: The filter the carrier is applied to.
        f_u: Frequency of the sinusoidal input.
        U: Amplitude of the sinusoidal input.
        n_carrier: Carrier periods the filter is settled over.
    """
    K = int(round(f0 / f_u))

    y = BinaryWave(np.arange(0, n_carrier + K + 2.5, 0.5) / f0).filter(h)

    t0 = n_carrier / f0

    # - A filter without DC feedback offsets the response to the carrier
    t = t0 + np.linspace(0, 1 / f0, 1001)
    y0 = (np.max(y(t)) + np.min(y(t))) / 2

    u = SineWave([0, U], f_u)

    def e(t):
        return u(t - t0) - (y(t) - y0)

    quantizer = QuantizerDelayHysteresis(0.0, 0.0)
    quantizer.t_step = 0.1 / f0
    quantizer.v = +1.0 if e(t0) > 0 else -1.0

    s = quantizer.v

    t = t0
    edges = []

    while True:
        t += quantizer.next(lambda dt, t=t: e(t + dt))

        if t >= t0 + K / f0:
            break

        edges.append(t - t0)

    assert len(edges) == 2 * K, "Simulation lost a transition."

    return np.array(edges), s


def demodulate(edges, s, f_u, U, n_settle=8, n_period=4, order=10):
    """
    Taylor coefficients of the average output, obtained by demodulating the output of the
    modulator and reading the fundamental and the third harmonic from its discrete
    Fourier transform.

    Arguments:
        edges: Transitions over one period of the input.
        s: Value of the output at the start of that period.
        f_u: Frequency of the sinusoidal input.
        U: Amplitude of the sinusoidal input.
        n_settle: Input periods discarded to settle the demodulation filter.
        n_period: Input periods the transform is taken over.
        order: Order of the demodulation filter.
    """
    v = BinaryWave(
        (edges[None, :] + np.arange(n_settle + n_period)[:, None] / f_u).ravel(), E=-s
    )

    # - Corner frequency of the demodulation filter and sampling frequency
    f_c = 4 * f_u
    f_s = 16 * f_c

    h = FilterButterworth(order, f_c)

    t = n_settle / f_u + np.arange(0.0, n_period / f_u - 0.5 / f_s, 1 / f_s)

    V = np.fft.rfft(v.filter(h)(t)) / (len(t) / 2)

    # - The demodulation filter attenuates the third harmonic slightly more than the
    #   fundamental, which is compensated for
    H = h.frequency_response(2 * np.pi * np.array([f_u, 3 * f_u]))

    V_1 = np.abs(V[n_period] / H[0])
    V_3 = np.abs(V[3 * n_period] / H[1])

    return V_1 / U, 4 * V_3 / U**3

In [ ]:
# - Amplitude of the sinusoidal input
U = 0.1
# - Carrier periods per input period, and the resulting input frequencies
K = np.unique(np.round(np.logspace(1, 4, 16)).astype(int))
f_u = f0 / K
# - Ratio of the corner frequency of the low-pass filter to the switching frequency
beta_u = 1e-2
# - Carrier periods the filters are settled over, forty time constants for the low-pass
#   filter and a single period for the integrator, which settles instantly
n_lpf = int(np.ceil(40 / (2 * np.pi * beta_u))) + 2

c_fu = np.zeros((4, len(K)))

for i, K_i in enumerate(K):
    edges, s = modulate(FilterIntegrator(2 * np.pi * f0 * 2 / np.pi), f0 / K_i, U, 1)

    W_1, W_3 = demodulate(edges, s, f0 / K_i, U)

    c_fu[1, i] = oip3(W_1, W_3)

    edges, s = modulate(
        FilterFirstOrder(beta_u * f0, 2 / (np.pi * beta_u)), f0 / K_i, U, n_lpf
    )

    W_1, W_3 = demodulate(edges, s, f0 / K_i, U)

    c_fu[2, i] = oip3(W_1, W_3)

c_fu[0] = f_u / f0
c_fu[3] = oip3(1.0, np.pi**2 * beta_u**2 / 12)

np.savetxt(
    out / "pwm-oip3-frequency.csv",
    c_fu.T,
    delimiter=",",
    header="fu_over_f0,OIP3_int,OIP3_lpf,OIP3_lpf_static",
    comments="",
)

In [ ]:
# - Slope of both curves, to detect a roll-off with the input frequency
for name, y in (("integrator", c_fu[1]), ("low-pass", c_fu[2])):
    slope = np.diff(y) / np.diff(np.log10(f_u / f0))

    print(
        f"{name:>10}: {y.min():6.1f} to {y.max():6.1f} dB, "
        f"slope {slope.min():+6.2f} to {slope.max():+6.2f} dB/dec"
    )

In [ ]:
plt.semilogx(f_u / f0, c_fu[1], "ko-", label=r"integrator")
plt.semilogx(f_u / f0, c_fu[2], "o-", color="gray", label=r"first-order low-pass")
plt.semilogx(f_u / f0, c_fu[3], "--", color="gray", label=r"$A_\mathrm{oip3}$ (static)")
plt.xlabel(r"$f_\mathrm{u}/f_0$")
plt.ylabel(r"$A_\mathrm{oip3,sim}$ [dB]")
plt.grid(which="both", ls="--", alpha=0.5)
plt.legend()
plt.show()